# Generating document id on page index

In [1]:
# Imports
import pageindex.utils as utils
import sys
from pathlib import Path

# Define root directory for notebook context
root_dir = Path.cwd().parent  # Va desde pipelines/ a la raiz del proyecto
sys.path.insert(0, str(root_dir))  # Agregar al path para importaciones

from core.config import settings
from core.client import gemini_client, pi_client

# SUBMIT DOC TO PAGE INDEX

In [2]:

pdf_path = root_dir / "data" / "raw_files" / "quechua_grammar.pdf"
doc_id = pi_client.submit_document(str(pdf_path))["doc_id"]
print('Document Submitted:', doc_id)

Document Submitted: pi-cml4je2es03tz0fqzdngcb4yf


# Creating the structure

In [4]:
import json

doc_id = "pi-cmkxfip4n02f80hpesma22pm9"
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of the Document:')
    utils.print_tree(tree)
    
    # guardar en un archivo JSON
    output_path = root_dir / "data" / "quechua_tree.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(tree, f, indent=2, ensure_ascii=False)
    print(f'\nTree saved to: {output_path}')
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': 'Modulo', 'node_id': '0000', 'summary': '# Modulo\n'},
 {'title': 'LENGUA QUECHUA II',
  'node_id': '0001',
  'prefix_summary': '# LENGUA QUECHUA II\n\n![img-0.jpeg](img-0...',
  'nodes': [{'title': 'INTRODUCCIÓN',
             'node_id': '0002',
             'summary': 'This module, Lengua Quechua II, is desig...'},
            {'title': 'INDICE', 'node_id': '0003', 'summary': '## INDICE\n'},
            {'title': 'CAPITULO I', 'node_id': '0004', 'summary': '## CAPITULO I\n'},
            {'title': 'MORFOLOGÍA QUECHUA',
             'node_id': '0005',
             'summary': 'The text covers basic concepts of Quechu...'},
            {'title': 'CAPITULO II',
             'node_id': '0006',
             'summary': 'This text, Chapter II, covers the Quechu...'},
            {'title': 'CAPITULO III',
             'node_id': '0007',
             'summary': 'Chapter III of the document focuses on t...'},
            {'title': 'CAPITULO 

In [5]:
utils.print_tree(tree)

[{'title': 'Modulo', 'node_id': '0000', 'summary': '# Modulo\n'},
 {'title': 'LENGUA QUECHUA II',
  'node_id': '0001',
  'prefix_summary': '# LENGUA QUECHUA II\n\n![img-0.jpeg](img-0...',
  'nodes': [{'title': 'INTRODUCCIÓN',
             'node_id': '0002',
             'summary': 'This module, Lengua Quechua II, is desig...'},
            {'title': 'INDICE', 'node_id': '0003', 'summary': '## INDICE\n'},
            {'title': 'CAPITULO I', 'node_id': '0004', 'summary': '## CAPITULO I\n'},
            {'title': 'MORFOLOGÍA QUECHUA',
             'node_id': '0005',
             'summary': 'The text covers basic concepts of Quechu...'},
            {'title': 'CAPITULO II',
             'node_id': '0006',
             'summary': 'This text, Chapter II, covers the Quechu...'},
            {'title': 'CAPITULO III',
             'node_id': '0007',
             'summary': 'Chapter III of the document focuses on t...'},
            {'title': 'CAPITULO IV',
             'node_id': '0008',
      

# Reasoning-Based Retrieval with Tree Search

In [4]:
# Cargar el árbol desde el JSON guardado
import json

with open(root_dir / "data" / "quechua_tree.json", 'r', encoding='utf-8') as f:
    tree = json.load(f)

print(f"Tree loaded with {len(tree)} nodes")

Tree loaded with 2 nodes


## Step 1: Tree Search - Identificar nodos relevantes

In [5]:
# Definir la consulta sobre gramática quechua
query = "¿Cuáles son las reglas de formación de plural en quechua?"

# Eliminar el campo 'text' del árbol para el prompt (solo usar títulos y resúmenes)
tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

# Usar Gemini para búsqueda en el árbol (await porque es async)
tree_search_result = await gemini_client.generate_text(search_prompt)
print("Tree Search Result:")
print(tree_search_result)

Tree Search Result:
{
    "thinking": "The question asks for the rules of plural formation in Quechua. Pluralization in Quechua involves specific suffixes depending on whether the word is a noun (nominal) or a verb (verbal). \n\n1.  **Nominal Pluralization:** Node `0015` (under Chapter I: Morphology) discusses 'Dependent nominal suffixes' and explicitly mentions 'flexive suffixes for ... number (singular and plural forms)'. This is the primary location for rules regarding noun pluralization (e.g., the suffix -kuna).\n2.  **Verbal Pluralization:** Node `0023` is titled 'Flexión de número' (Inflection of number) and its summary states it details 'verb conjugations ... covering number (pluralization)'. This explains how verbs are pluralized (e.g., using -chik, -ku).\n3.  **Pronoun Pluralization:** Node `0028` (Chapter II) focuses on pronouns and mentions 'Special attention is given to the nuances of plural forms for personal and possessive pronouns'. While 0015 covers general nominal rule

## Step 2: Mostrar nodos recuperados y razonamiento

In [7]:
# Crear mapa de nodos para acceso rápido
node_map = utils.create_node_mapping(tree)

# Parsear resultado JSON
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks for the rules of plural formation in Quechua. Pluralization in Quechua involves
specific suffixes depending on whether the word is a noun (nominal) or a verb (verbal).

1.  **Nominal Pluralization:** Node `0015` (under Chapter I: Morphology) discusses 'Dependent
nominal suffixes' and explicitly mentions 'flexive suffixes for ... number (singular and plural
forms)'. This is the primary location for rules regarding noun pluralization (e.g., the suffix
-kuna).
2.  **Verbal Pluralization:** Node `0023` is titled 'Flexión de número' (Inflection of number) and
its summary states it details 'verb conjugations ... covering number (pluralization)'. This explains
how verbs are pluralized (e.g., using -chik, -ku).
3.  **Pronoun Pluralization:** Node `0028` (Chapter II) focuses on pronouns and mentions 'Special
attention is given to the nuances of plural forms for personal and possessive pronouns'. While 0015
covers general nominal rules, 0028 covers specific p

## Step 3: Extraer contexto relevante de los nodos

In [9]:
# Extraer el texto completo de los nodos relevantes
node_list = tree_search_result_json["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1500] + '...' if len(relevant_content) > 1500 else relevant_content)

Retrieved Context:

6

# MORFOLOGÍA QUECHUA

## 1.1. CONCEPTOS BÁSICOS:

La Morfología.

Es la parte de la gramática que estudia la constitución de las palabras. Las palabras de la lengua
quechua, a diferencia de otras lenguas del mundo y en especial del castellano, están conformadas por
diversos elementos que están determinados por la tipología aglutinante o sufijante.

## 1.2. ESTUDIO DE LA PALABRA QUECHUA:

### 1.2.1. ¿Qué es la palabra?

La palabra es un conjunto de fonemas que porta un significado, es decir, sistema de fonemas que hace
conocer algo. La palabra es un conjunto de sonidos articulados, que podemos representar gráficamente
con letras, a los que asociamos un significado: árbol, río. “Toda palabra tiene un significante
(forma) y un significado (contenido)”.

### 1.2.2. Clases de palabras

Las palabras se clasifican en distintos grupos, dependiendo de los criterios que se establece.
Veamos en seguida la siguiente clasificación.

#### a) Teniendo en cuenta su origen y estr

## Step 4: Generar respuesta basada en el contexto

In [10]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await gemini_client.generate_text(answer_prompt)
utils.print_wrapped(answer)

# Guardar la respuesta en un archivo de texto
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = root_dir / "data" / f"quechua_answer_{timestamp}.txt"

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(f"Query: {query}\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*80 + "\n\n")
    f.write("Answer:\n")
    f.write(answer)
    f.write("\n\n" + "="*80 + "\n")
    f.write(f"Retrieved Nodes ({len(node_list)}):\n")
    for node_id in node_list:
        node = node_map[node_id]
        f.write(f"- Node {node['node_id']} (Page {node['page_index']}): {node['title']}\n")

print(f'\nAnswer saved to: {output_file}')

Generated Answer:

Según el texto proporcionado, las reglas de formación del plural (flexión de número) en quechua son
las siguientes:

1.  **Para los nombres (sustantivos):**
    *   El singular no lleva ningún sufijo específico.
    *   El plural lleva el sufijo pluralizador **–kuna**.
    *   *Ejemplos:* Chukcha–kuna (cabellos), Qallu–kuna (lenguas).

2.  **Para pluralizar personas:**
    Se utilizan marcas o sufijos específicos dependiendo de la persona gramatical:
    *   **-nchik:** Primera persona inclusiva.
    *   **-yku:** Primera persona exclusiva.
    *   **-chik:** Segunda persona.
    *   **-ku:** Tercera persona.

Answer saved to: w:\PROFESIONAL\page-index-test\data\quechua_answer_20260201_211607.txt


# Procesando nuevo documento: Gramar

In [2]:
# Subir el nuevo documento gramar a Page Index
gramar_pdf_path = root_dir / "data" / "raw_files" / "gramar.pdf"
gramar_doc_id = pi_client.submit_document(str(gramar_pdf_path))["doc_id"]
print('Gramar Document Submitted:', gramar_doc_id)

Gramar Document Submitted: pi-cml4jndqk0cbj09qzqxxg8vba


## Generar estructura del documento Gramar

In [ ]:
# Reemplaza con el doc_id generado en la celda anterior
gramar_doc_id = "pi-cml4jndqk0cbj09qzqxxg8vba"

if pi_client.is_retrieval_ready(gramar_doc_id):
    gramar_tree = pi_client.get_tree(gramar_doc_id, node_summary=True)['result']
    print('Simplified Tree Structure of Gramar Document:')
    utils.print_tree(gramar_tree)
    
    # Guardar en un archivo JSON
    gramar_output_path = root_dir / "data" / "gramar_tree.json"
    with open(gramar_output_path, 'w', encoding='utf-8') as f:
        json.dump(gramar_tree, f, indent=2, ensure_ascii=False)
    print(f'\nGramar tree saved to: {gramar_output_path}')
else:
    print("Processing document, please try again later...")

# Traducción de frases en Quechua usando ambos documentos

In [ ]:
# Cargar ambos árboles JSON
with open(root_dir / "data" / "quechua_tree.json", 'r', encoding='utf-8') as f:
    quechua_tree = json.load(f)

with open(root_dir / "data" / "gramar_tree.json", 'r', encoding='utf-8') as f:
    gramar_tree = json.load(f)

print(f"Quechua tree loaded: {len(quechua_tree)} nodes")
print(f"Gramar tree loaded: {len(gramar_tree)} nodes")

## Step 1: Tree Search en ambos documentos

In [ ]:
# Frases en quechua para traducir
translation_query = """
Traduce las siguientes 5 frases del quechua al español:
1. Ñuqa hatun wasipi tiyani
2. Qam imaynataq kasanki
3. Paykunaqa chakrapi llank'anku
4. Inti rikcharimuchkan
5. Mama wayk'usqanta mikhuni
"""

# Preparar ambos árboles sin el campo 'text'
quechua_tree_without_text = utils.remove_fields(quechua_tree.copy(), fields=['text'])
gramar_tree_without_text = utils.remove_fields(gramar_tree.copy(), fields=['text'])

# Prompt combinado para buscar en ambos documentos
combined_search_prompt = f"""
You are given a question and two tree structures from different documents about Quechua language.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes from BOTH documents that are likely to contain the answer to the question.

Question: {translation_query}

Document 1 - Quechua Grammar tree structure:
{json.dumps(quechua_tree_without_text, indent=2)}

Document 2 - Gramar tree structure:
{json.dumps(gramar_tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes from both documents are relevant to the question>",
    "quechua_nodes": ["node_id_1", "node_id_2", ...],
    "gramar_nodes": ["node_id_1", "node_id_2", ...]
}}
Directly return the final JSON structure. Do not output anything else.
"""

# Buscar en ambos árboles
translation_search_result = await gemini_client.generate_text(combined_search_prompt)
print("Translation Search Result:")
print(translation_search_result)

## Step 2: Mostrar nodos recuperados de ambos documentos

In [ ]:
# Crear mapas de nodos para ambos documentos
quechua_node_map = utils.create_node_mapping(quechua_tree)
gramar_node_map = utils.create_node_mapping(gramar_tree)

# Parsear resultado JSON
translation_result_json = json.loads(translation_search_result)

print('Reasoning Process:')
utils.print_wrapped(translation_result_json['thinking'])

print('\n--- Retrieved Nodes from Quechua Grammar Document ---')
for node_id in translation_result_json.get("quechua_nodes", []):
    node = quechua_node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

print('\n--- Retrieved Nodes from Gramar Document ---')
for node_id in translation_result_json.get("gramar_nodes", []):
    node = gramar_node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

## Step 3: Extraer y combinar contexto de ambos documentos

In [ ]:
# Extraer contexto de ambos documentos
quechua_nodes_list = translation_result_json.get("quechua_nodes", [])
gramar_nodes_list = translation_result_json.get("gramar_nodes", [])

quechua_content = "\n\n".join(quechua_node_map[node_id]["text"] for node_id in quechua_nodes_list)
gramar_content = "\n\n".join(gramar_node_map[node_id]["text"] for node_id in gramar_nodes_list)

# Combinar el contenido relevante
combined_content = f"""
=== Context from Quechua Grammar Document ===
{quechua_content}

=== Context from Gramar Document ===
{gramar_content}
"""

print('Combined Retrieved Context (preview):')
utils.print_wrapped(combined_content[:2000] + '...' if len(combined_content) > 2000 else combined_content)

## Step 4: Generar traducción usando contexto combinado

In [ ]:
translation_prompt = f"""
Answer the question based on the context from both documents:

Question: {translation_query}

Context: {combined_content}

Provide clear, accurate translations for each of the 5 Quechua phrases based only on the context provided.
Include explanations of the grammatical structures used when relevant.
"""

print('Generated Translation:\n')
translation_answer = await gemini_client.generate_text(translation_prompt)
utils.print_wrapped(translation_answer)

# Guardar la traducción en un archivo de texto
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
translation_output_file = root_dir / "data" / f"quechua_translation_{timestamp}.txt"

with open(translation_output_file, 'w', encoding='utf-8') as f:
    f.write("Translation Query:\n")
    f.write(translation_query)
    f.write("\n" + "="*80 + "\n\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("Translation Result:\n")
    f.write(translation_answer)
    f.write("\n\n" + "="*80 + "\n")
    f.write(f"Sources Used:\n")
    f.write(f"\nFrom Quechua Grammar ({len(quechua_nodes_list)} nodes):\n")
    for node_id in quechua_nodes_list:
        node = quechua_node_map[node_id]
        f.write(f"- Node {node['node_id']} (Page {node['page_index']}): {node['title']}\n")
    f.write(f"\nFrom Gramar ({len(gramar_nodes_list)} nodes):\n")
    for node_id in gramar_nodes_list:
        node = gramar_node_map[node_id]
        f.write(f"- Node {node['node_id']} (Page {node['page_index']}): {node['title']}\n")

print(f'\nTranslation saved to: {translation_output_file}')